# 06. RFM-сегментация

До этого мы делили клиентов по одному признаку (размер первого чека). Это полезно, но грубо. RFM делит клиентов сразу по трём осям:

- R (Recency): сколько дней прошло с последней покупки. Чем меньше, тем 'свежее' клиент.
- F (Frequency): сколько всего у него было заказов. Чем больше, тем активнее.
- M (Monetary): сколько он потратил суммарно. Чем больше, тем ценнее в деньгах.

По каждой оси разбиваем клиентов на квинтили (1 хуже, 5 лучше) и получаем что-то вроде кода '555' (топ по всем трём) или '111' (хуже всех). Поверх этих кодов накладываем понятные бизнес-имена сегментов: 'Чемпионы', 'Лояльные', 'Под угрозой', 'Потерянные'.

Главная польза RFM в том, что она позволяет к каждому сегменту привязать осмысленное действие, а не общую кампанию на всю базу.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (10, 5)

df = pd.read_parquet('../data/clean.parquet')
print(f'Транзакций: {len(df):,}')

## Считаем R, F, M для каждого клиента

Точкой отсчёта для recency возьмём дату последней транзакции в датасете плюс один день. То есть притворимся, что 'сегодня' это завтра после конца наблюдений. Это стандартный приём, чтобы не получить нулевые значения у самых свежих клиентов.

In [ ]:
snapshot_date = df['InvoiceDate'].max() + pd.Timedelta(days=1)
print(f'Snapshot date: {snapshot_date:%Y-%m-%d}')

rfm = (
    df.groupby('Customer ID')
    .agg(
        recency=('InvoiceDate', lambda s: (snapshot_date - s.max()).days),
        frequency=('Invoice', 'nunique'),
        monetary=('Revenue', 'sum'),
    )
    .reset_index()
)
rfm.head()

## Присваиваем баллы 1-5 по каждой оси

Здесь есть тонкость с recency: маленькое значение это хорошо (клиент только что купил), поэтому шкалу для R инвертируем.

In [ ]:
# qcut делит на квинтили по содержимому. У F часто много дублей (большая часть клиентов с 1-2 заказами),
# поэтому используем rank(method='first'), чтобы избежать ошибки 'bin edges must be unique'.
rfm['R_score'] = pd.qcut(rfm['recency'].rank(method='first'), 5, labels=[5, 4, 3, 2, 1]).astype(int)
rfm['F_score'] = pd.qcut(rfm['frequency'].rank(method='first'), 5, labels=[1, 2, 3, 4, 5]).astype(int)
rfm['M_score'] = pd.qcut(rfm['monetary'].rank(method='first'), 5, labels=[1, 2, 3, 4, 5]).astype(int)
rfm['RFM_score'] = rfm['R_score'].astype(str) + rfm['F_score'].astype(str) + rfm['M_score'].astype(str)
rfm.head()

## Превращаем коды в бизнес-сегменты

Существует много вариантов разметки RFM-сегментов, я возьму распространённую и понятную схему. Основные правила:
- Recency хороший И Frequency высокий → клиент в активной фазе.
- Recency плохой, но раньше Frequency был высокий → раньше любил, теперь забыл, надо возвращать.
- Recency и Frequency низкие → 'потерянный', работа с ним обычно не окупается.

In [ ]:
def assign_segment(row):
    r, f, m = row['R_score'], row['F_score'], row['M_score']
    if r >= 4 and f >= 4:
        return 'Чемпионы'
    if r >= 3 and f >= 3:
        return 'Лояльные'
    if r >= 4 and f <= 2:
        return 'Новички'
    if r == 3 and f <= 2:
        return 'Перспективные'
    if r <= 2 and f >= 4:
        return 'Под угрозой'
    if r <= 2 and f >= 2:
        return 'Спящие'
    return 'Потерянные'

rfm['segment'] = rfm.apply(assign_segment, axis=1)

segment_summary = (
    rfm.groupby('segment')
    .agg(
        n_customers=('Customer ID', 'count'),
        avg_recency_days=('recency', 'mean'),
        avg_frequency=('frequency', 'mean'),
        avg_monetary=('monetary', 'mean'),
        total_monetary=('monetary', 'sum'),
    )
    .round(1)
    .sort_values('total_monetary', ascending=False)
)
segment_summary['share_revenue_pct'] = (segment_summary['total_monetary'] / segment_summary['total_monetary'].sum() * 100).round(1)
segment_summary

Что бросается в глаза по итоговой таблице:
- 'Чемпионов' обычно немного, но они дают непропорционально большую долю выручки. Это та самая голова распределения.
- 'Под угрозой' это сегмент, в котором клиент когда-то был активным и приносил деньги, но давно не покупал. Самый интересный для retention-кампаний.
- 'Потерянные' это сегмент, на который не стоит тратить маркетинговый бюджет. Деньги уйдут в никуда.

In [ ]:
# Сводный график: размер сегмента vs его доля в выручке
fig, ax = plt.subplots(figsize=(11, 6))
x = np.arange(len(segment_summary))
width = 0.4
share_customers = segment_summary['n_customers'] / segment_summary['n_customers'].sum() * 100
share_revenue = segment_summary['share_revenue_pct']

ax.bar(x - width/2, share_customers, width, label='доля клиентов, %', color='#4C72B0')
ax.bar(x + width/2, share_revenue, width, label='доля выручки, %', color='#55A868')
ax.set_xticks(x)
ax.set_xticklabels(segment_summary.index, rotation=20)
ax.set_ylabel('%')
ax.set_title('RFM-сегменты: доля клиентов и доля выручки')
ax.legend()
plt.tight_layout()
plt.savefig('../images/rfm_segments.png', dpi=120, bbox_inches='tight')
plt.show()

Идея графика простая: где зелёная палка сильно выше синей, там сегмент даёт выручки больше, чем составляет от базы. Это и есть 'ценные' сегменты, на которые имеет смысл фокусировать ресурсы.

Где синяя палка выше зелёной, там сегмент по объёму большой, но малоценный. Сюда массовые скидки лить нельзя, окупаемость провалится.

## Что делать с каждым сегментом

Это не теоретическая разметка, а готовый бриф для CRM-команды:

- Чемпионы: программы лояльности, ранний доступ к новинкам, никаких массовых скидок (они и так покупают, скидки только режут маржу).
- Лояльные: апсейл по сопутствующим категориям, предложение членства или подписки.
- Под угрозой: персональная реактивация. Здесь триггерные письма работают лучше всего, потому что клиент ещё помнит бренд.
- Спящие: широкая реактивационная кампания с бо́льшей скидкой, но без таргетинга по интересам, потому что данных по последним предпочтениям мало.
- Новички: онбординг, акцент на UX и быстрой второй покупке. Здесь как раз сценарий, который мы расписывали в ноутбуке 03.
- Перспективные: подталкиваем к третьей-четвёртой покупке, чтобы перешли в 'Лояльных'.
- Потерянные: не трогаем. Если общая база не растёт, можно один раз протестировать win-back с очень дешёвым контентным письмом, без скидок.

## Сохраняем результаты

In [ ]:
rfm.to_parquet('../data/rfm.parquet', index=False)
print('Сохранено: data/rfm.parquet')

## Итог по RFM

RFM это не модель и не статистика, а инструмент сегментирования базы по трём поведенческим осям. Главное достоинство в том, что результат можно сразу пустить в работу: к каждому сегменту привязывается понятное действие. Главный риск в том, что границы квинтилей чувствительны к данным: если структура трафика поменялась, шкалы надо пересобирать. На практике RFM-таблицу пересчитывают раз в неделю или в месяц.